In [1]:
import os
import time
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import spacy
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Download spacy model (Google Colab ke liye)
os.system("python -m spacy download en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# 2. Make Folder for Figures
os.makedirs("figures", exist_ok=True)

# 3. Web Scraping (Code khud internet se file banayega)
urls = [
    'https://techncruncher.blogspot.com/2025/01/top-10-ai-tools-that-will-transform.html',
    'https://techncruncher.blogspot.com/2023/12/limewire-ai-studio-review-2023-details.html',
    'https://techncruncher.blogspot.com/2023/01/top-10-ai-tools-in-2023-that-will-make.html',
    'https://techncruncher.blogspot.com/2022/11/top-10-ai-content-generator-writer.html',
    'https://techncruncher.blogspot.com/2022/09/cj-affiliate-ultimate-guide-to.html'
]

headers = {"User-Agent": "Mozilla/5.0"}
articles = []

print("Scraping started... Please wait.")
for url in urls:
    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.content, 'lxml')
        title = soup.find('h3', class_='post-title')
        title = title.get_text(strip=True) if title else soup.title.get_text(strip=True)
        content_div = soup.find('div', class_='post-body entry-content')
        if not content_div:
            continue
        text = ' '.join(p.get_text(strip=True) for p in content_div.find_all('p'))
        if len(text) > 100:
            articles.append({'url': url, 'title': title, 'text': text})
        time.sleep(1)
    except:
        pass

df = pd.DataFrame(articles)
df.to_csv("articles_raw.csv", index=False)
print("Scraping done! Raw data saved.")

# 4. Feature Engineering (spaCy & TextBlob)
def extract_features(text):
    doc = nlp(text)
    blob = TextBlob(text)
    return pd.Series({
        'num_tokens': len(doc),
        'num_sentences': len(list(doc.sents)),
        'num_entities': len(doc.ents),
        'num_nouns': len([t for t in doc if t.pos_ == 'NOUN']),
        'sentiment_polarity': blob.sentiment.polarity,
        'sentiment_subjectivity': blob.sentiment.subjectivity
    })

print("Extracting features... This might take a minute.")
features = df['text'].apply(extract_features)
df = pd.concat([df, features], axis=1)
df['title_length'] = df['title'].apply(len)

# Save Final Dataset
df.to_csv("articles.csv", index=False)
print("Feature extraction done! 'articles.csv' is ready.")

# 5. Exploratory Data Analysis (Graphs)
plt.style.use('ggplot')

# Graph 1
plt.figure(figsize=(8, 4))
sns.histplot(df['title_length'], kde=True, color='skyblue')
plt.title('Title Length Distribution')
plt.savefig('figures/title_length_distribution.png')
plt.close()

# Graph 2
plt.figure(figsize=(8, 4))
sns.histplot(df['sentiment_polarity'], kde=True, color='lightcoral')
plt.title('Sentiment Polarity Distribution')
plt.savefig('figures/sentiment_polarity_distribution.png')
plt.close()

# Graph 3
numeric_cols = ['title_length', 'num_tokens', 'num_sentences', 'num_entities', 'sentiment_polarity']
g = sns.pairplot(df[numeric_cols])
g.savefig('figures/pairplot_features.png')
plt.close()

# Graph 4
vectorizer = TfidfVectorizer(stop_words='english', max_features=20)
X = vectorizer.fit_transform(df['text'])
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_sums = tfidf_df.sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=tfidf_sums.index, y=tfidf_sums.values)
plt.title("Top 20 TF-IDF Terms")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/tfidf_top20_terms.png')
plt.close()

print("All tasks completed successfully! Check the folder icon on the left to download your files.")

Scraping started... Please wait.
Scraping done! Raw data saved.
Extracting features... This might take a minute.
Feature extraction done! 'articles.csv' is ready.
All tasks completed successfully! Check the folder icon on the left to download your files.
